## Notebook to demo Point In Polygon analysis where polygons are in a shapefile.

In [ ]:
# from spark_esri import spark_start, spark_stop

In [ ]:
import geofunctions as S
import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz

In [ ]:
# config = {
#     "spark.driver.memory":"32G",
#     "spark.executor.memory":"32G",
#     "spark.jars":"geofunctions-0.6.jar",
# }

# spark_stop()
# spark = spark_start(config=config)

In [ ]:
cell = 10.0  # QR in degrees.
dist = 0.00001  # Padding in degrees.

In [ ]:
xmin, ymin, xmax, ymax = (-180.0, -80.0, 180.0, 80.0)
xdel = xmax - xmin
ydel = ymax - ymin

lhs = (
    spark.range(0, 10_000)
    .withColumn("x", F.rand() * xdel + xmin)
    .withColumn("y", F.rand() * ydel + ymin)
    .select(
        S.st_point("x", "y").alias("pt"),
        S.qr_fromxy("x", "y", cell).alias("qr"),
    )
)

In [ ]:
rhs = (
    spark.read.format("shp")
    .options(path="../data/world/tz_world.shp")
    .load()
    .withColumn("qr", F.explode(S.qr_clip("shape", cell, dist)))
    .drop("shape")
    .select("tzid", "qr.qr", "qr.geom")
)

In [ ]:
df = (
    lhs.join(rhs, "qr")
    .drop("qr")
    .filter(S.st_contains("geom", "pt"))
    .drop("geom")
)

In [ ]:
pdf = df.withColumnRenamed("pt","geometry").toPandas()
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs="EPSG:4326")
gdf.sindex;  # Enable the spatial index.

In [ ]:
gdf.explore(tiles=xyz.Esri.WorldGrayCanvas)